# 02 · Parámetros derivados y clasificación planetaria

**Objetivo.** Añadir al catálogo limpio las cantidades derivadas que usan los tres análisis.

**Entrada:** `data_processed/02_catalogo_limpio_base.csv`

**Salidas:**
- `data_processed/03_catalogo_con_parametros_derivados.csv`
- Tablas D-T6 (clasificaciones auxiliares) y D-T7 (verificación de los parámetros derivados)

| Cantidad | Definición |
|---|---|
| `densidad_calculada` | $\bar\rho = \rho_\oplus\,(M_p/M_\oplus)/(R_p/R_\oplus)^3$, con $\rho_\oplus = 5.51$ g/cm³ |
| `clase_radio` | terrestre ($R_p<1.25$), supertierra ($1.25$–$2$), neptuniano ($2$–$6$), gigante_gaseoso ($6$–$15$), mayor_que_jupiter ($\geq 15\,R_\oplus$) |
| `regimen_planetario` | super_tierra ($1<R_p<1.6$), valle_excluido ($1.6\leq R_p\leq 2.0$), sub_neptuno ($2.0<R_p<4\,R_\oplus$) |
| `sub_neptune_conservador` | 1 = sub-Neptuno, 0 = super-Tierra, vacío en el valle excluido |
| `a_final_au` | semieje mayor del catálogo o, si falta, $a=[M_\star (P/\mathrm{año})^2]^{1/3}$ |
| `insol_calculada` | $F/F_\oplus = R_\star^2\,(T_{\rm eff}/5772\,\mathrm{K})^4/a^2$ |

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from funciones import *

usar_estilo()

In [2]:
df_clean = pd.read_csv(CATALOGO_LIMPIO)
df = agregar_derivadas(df_clean)
print("Catálogo con parámetros derivados:", df.shape)
df[["pl_name", "pl_rade", "pl_bmasse", "densidad_calculada", "clase_radio",
    "regimen_planetario", "a_final_au", "a_origen", "log_insol_calculada"]].head()

Catálogo con parámetros derivados: (6267, 54)


,pl_name,pl_rade,pl_bmasse,densidad_calculada,clase_radio,regimen_planetario,a_final_au,a_origen,log_insol_calculada
0,11 Com b,12.2,4914.898486,14.913743,gigante_gaseoso,NaN,1.178,catalogado,1.841183
1,11 UMi b,12.3,4684.814200,13.871666,gigante_gaseoso,NaN,1.530,catalogado,2.031819
2,14 And b,13.1,1131.151301,2.772416,gigante_gaseoso,NaN,0.775,catalogado,2.057780
3,14 Her b,12.5,2828.672822,7.980025,gigante_gaseoso,NaN,2.839,catalogado,-1.105156
4,16 Cyg B b,13.5,565.737400,1.266967,gigante_gaseoso,NaN,1.660,catalogado,-0.340693


## 1. Clasificaciones

La clasificación principal es por radio (`clase_radio`); se describe en el notebook 03. Las clasificaciones por masa y período y los indicadores son auxiliares.

In [3]:
definiciones = {
    "clase_masa": {"masa_terrestre": "Mp < 2 M_Tierra", "supertierra_masa": "2 <= Mp < 10",
                   "subneptuno_neptuno_masa": "10 <= Mp < 50", "gigante_masa": "50 <= Mp < 4000",
                   "muy_masivo": "Mp >= 4000", "sin_dato": "masa no disponible"},
    "clase_periodo": {"USP": "P < 1 d", "periodo_corto": "1 <= P < 10 d",
                      "periodo_intermedio": "10 <= P < 100 d", "periodo_largo": "P >= 100 d",
                      "sin_dato": "período no disponible"},
    "indicador": {"es_pequeno": "Rp < 2 R_Tierra", "es_gigante": "Rp >= 6 R_Tierra",
                  "es_hot_jupiter": "Rp >= 6 R_Tierra y P < 10 d",
                  "es_super_puff": "Rp >= 4 R_Tierra y densidad < 0.3 g/cm^3"},
}
filas = []
for clasificacion, clases in definiciones.items():
    for clase, criterio in clases.items():
        n = int(df[clase].sum()) if clasificacion == "indicador" else int((df[clasificacion] == clase).sum())
        filas.append({"clasificacion": clasificacion, "clase": clase, "criterio": criterio,
                      "N_planetas": n, "porcentaje": 100 * n / len(df)})
tabla_auxiliares = guardar_tabla(
    pd.DataFrame(filas), "D-T6", "clasificaciones_auxiliares",
    titulo="Clasificaciones auxiliares por masa y período orbital, e indicadores de subpoblaciones.",
    seccion="6.1.1", destino="anexo", decimales=2,
    nota="Las masas y radios no medidos son estimaciones del archivo a partir de una relación masa-radio.",
    encabezados={"clasificacion": "Clasificación", "clase": "Clase"})
tabla_auxiliares

Tabla D-T6 guardada: tablas/D-T6_clasificaciones_auxiliares.csv


,clasificacion,clase,criterio,N_planetas,porcentaje
0,clase_masa,masa_terrestre,Mp < 2 M_Tierra,507,8.09
1,clase_masa,supertierra_masa,2 <= Mp < 10,2741,43.74
2,clase_masa,subneptuno_neptuno_masa,10 <= Mp < 50,950,15.16
3,clase_masa,gigante_masa,50 <= Mp < 4000,1879,29.98
4,clase_masa,muy_masivo,Mp >= 4000,164,2.62
5,clase_masa,sin_dato,masa no disponible,26,0.41
6,clase_periodo,USP,P < 1 d,159,2.54
7,clase_periodo,periodo_corto,1 <= P < 10 d,2695,43.00
8,clase_periodo,periodo_intermedio,10 <= P < 100 d,2100,33.51
9,clase_periodo,periodo_largo,P >= 100 d,973,15.53


## 2. Verificación de los parámetros derivados

Se compara el semieje mayor derivado con la tercera ley de Kepler y el flujo incidente calculado con los valores del catálogo, en los planetas que tienen ambos.

In [4]:
def comparar(nombre, catalogado, calculado):
    ambos = catalogado.notna() & calculado.notna() & (catalogado > 0) & (calculado > 0)
    razon = catalogado[ambos] / calculado[ambos]
    return {"comparacion": nombre, "N": int(ambos.sum()), "razon_mediana": razon.median(),
            "razon_Q1": razon.quantile(0.25), "razon_Q3": razon.quantile(0.75),
            "abs_log10_razon_mediana": np.log10(razon).abs().median()}

tabla_verificacion = pd.DataFrame([
    comparar("Semieje mayor: catalogado / Kepler", df["pl_orbsmax"], df["a_kepler_au"]),
    comparar("Flujo incidente: catalogado / calculado", df["pl_insol"], df["insol_calculada"]),
])
origen = df.loc[df["insol_calculada"].notna(), "a_origen"].value_counts()
tabla_verificacion = guardar_tabla(
    tabla_verificacion, "D-T7", "verificacion_parametros_derivados",
    titulo="Verificación de los parámetros derivados frente a los valores del catálogo.",
    seccion="6.1.1", destino="anexo",
    nota=(f"El flujo incidente se calculó para {int(df['insol_calculada'].notna().sum())} planetas; en "
          f"{int(origen.get('kepler_derivado', 0))} de ellos el semieje mayor se derivó con la tercera ley de Kepler."),
    encabezados={"comparacion": "Comparación", "razon_mediana": "Razón mediana", "razon_Q1": "$Q_1$",
                 "razon_Q3": "$Q_3$", "abs_log10_razon_mediana": r"$|\log_{10}$ razón$|$ mediana"})
tabla_verificacion

Tabla D-T7 guardada: tablas/D-T7_verificacion_parametros_derivados.csv


,comparacion,N,razon_mediana,razon_Q1,razon_Q3,abs_log10_razon_mediana
0,Semieje mayor: catalogado / Kepler,5513,1.0001,0.9944,1.0057,0.0024
1,Flujo incidente: catalogado / calculado,5644,0.9973,0.9193,1.0802,0.0351


## 3. Guardar

In [5]:
df.to_csv(CATALOGO_DERIVADO, index=False)
print("Guardado:", CATALOGO_DERIVADO.name, df.shape)
df["clase_radio"].value_counts()

Guardado: 03_catalogo_con_parametros_derivados.csv (6267, 54)


clase_radio
neptuniano           2374
gigante_gaseoso      1853
supertierra          1188
terrestre             571
mayor_que_jupiter     237
sin_dato               44
Name: count, dtype: int64